In [ ]:
!pip install --upgrade transformers

In [ ]:
from transformers import (
    WhisperForConditionalGeneration,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor
)
from transformers import WhisperForConditionalGeneration
model_name = 'openai/whisper-small'
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
tokenizer = WhisperTokenizer.from_pretrained(
    model_name,
    language = 'uz',
    task = 'transcribe'
)

processor = WhisperProcessor.from_pretrained(
    model_name,
    language = 'uz',
    task = 'transcribe'
)


model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.generation_config.language = 'uz'
model.generation_config.task = 'transcribe'


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
from datasets import load_dataset, Audio
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
dataset = load_dataset('murodbek/uzbek-speech-corpus', split="train[:10%]")
dataset = dataset.train_test_split(test_size = 0.1, seed = 42)
print(f"Training samples: {len(dataset['train'])}")
print(f"Testing samples: {len(dataset['test'])}")

dataset = dataset.cast_column('audio', Audio(sampling_rate = 16000))

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Training samples: 9069
Testing samples: 1008


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=16000
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("Preprocessing dataset...")
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names["train"]
)
print("Preprocessing complete")

Preprocessing dataset...


Map:   0%|          | 0/9069 [00:00<?, ? examples/s]

Map:   0%|          | 0/1008 [00:00<?, ? examples/s]

Preprocessing complete


In [ ]:
!pip install evaluate

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    model = model,
    tokenizer = processor.tokenizer
)

import evaluate
wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
  pred_ids = pred.predictions
  label_ids = pred.label_ids
  label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
  pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens = True)
  label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens = True)
  wer = wer_metric.compute(predictions = pred_str, references = label_str)
  cer = cer_metric.compute(predictions = pred_str, references = label_str)

  return{'wer': wer * 100, 'cer': cer * 100}


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, WhisperDataCollatorWithPadding
training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-uzbek',
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps = 2,
    num_train_epochs = 3,
    learning_rate = 1e-5,
    warmup_steps = 100,
    fp16 = True,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    logging_steps = 25,
    predict_with_generate = True,
    generation_max_length = 225,
    metric_for_best_model = 'wer',
    greater_is_better = False,
    save_total_limit =2,
    report_to = ['tensorboard'],
)

# Redefine data_collator with the correct class for Whisper
data_collator = WhisperDataCollatorWithPadding(processor=processor)

trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

# Training
print('Training started')
trainer.train()
print('Training complete.')

ImportError: cannot import name 'WhisperDataCollatorWithPadding' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [ ]:
# Evaluate
metrics = trainer.evaluate()
print(f"WER: {metrics['eval_wer']}%")
print(f"CER: {metrics['eval_cer']}%")
output = './uzbek-whisper'
model.save_pretrained(output)
processor.save_pretrained(output)
print(f"Model is saved in {output}")

# Testing on sample
sample = dataset['test'][0]
input_features = torch.tensor([sample['input_features']]).to('cuda')
predicted_ids = model.generate(input_features)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens = True)[0]
print(f"Sample transcription: {transcription}")
